# Action Recognition Thesis — Google Colab (Pro) Notebook

**Why Colab Pro:** up to ~24 h runtimes (vs Kaggle's 12 h), background execution, and bigger GPUs — so all three models can train in one session.

**Before running:**
1. **Runtime → Change runtime type → GPU** (pick **A100** or **L4** if available; T4 also works).
2. You need a **Kaggle API token** (one-time): kaggle.com → your avatar → **Settings → API → Create New Token** → downloads `kaggle.json`. Cell 3 will ask you to upload it (and caches it to Drive for next time).

Then **Runtime → Run all**. All logic lives in `run_all.py` in the repo, so this notebook never needs editing.
Results persist to your Google Drive, so a dropped session resumes where it left off.

In [ ]:
# ── Cell 1: GPU check + mount Google Drive ──────────────────────────────────
import torch
print(f'PyTorch {torch.__version__}  CUDA={torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'  {p.name}  ({p.total_memory/1e9:.1f} GB)')

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'decord', 'fvcore', 'scikit-learn', 'huggingface_hub', 'kaggle'
], check=True)
print('deps installed')

In [ ]:
# ── Cell 3: Kaggle API token (uploads once, then cached in Drive) ────────────
import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
drive_token = '/content/drive/MyDrive/kaggle.json'
local_token = '/root/.kaggle/kaggle.json'

if os.path.exists(drive_token):
    shutil.copy(drive_token, local_token)
    print('Using kaggle.json cached in Drive.')
elif not os.path.exists(local_token):
    from google.colab import files
    print('Upload kaggle.json  (Kaggle -> Settings -> API -> Create New Token):')
    up = files.upload()
    shutil.move(next(iter(up)), local_token)
    shutil.copy(local_token, drive_token)   # cache for future sessions
    print('Saved kaggle.json to Drive for next time.')
os.chmod(local_token, 0o600)
print('kaggle configured')

In [ ]:
# ── Cell 4: Get UCF101 (Drive-cached zip -> unzip to fast local disk) ────────
import os, glob, shutil, subprocess

DATA_DIR  = '/content/ucf101'                              # fast local SSD (train reads here)
DRIVE_ZIP = '/content/drive/MyDrive/ucf101_dataset.zip'    # cached download
SLUG = 'matthewjansen/ucf101-action-recognition'

if not (os.path.isdir(DATA_DIR) and os.listdir(DATA_DIR)):
    if not os.path.exists(DRIVE_ZIP):
        print('Downloading UCF101 from Kaggle (~6.5 GB, one time)...')
        subprocess.run(['kaggle', 'datasets', 'download', '-d', SLUG, '-p', '/content', '--quiet'], check=True)
        local_zip = glob.glob('/content/ucf101-action-recognition*.zip')[0]
        print('Caching zip to Drive for future sessions...')
        shutil.copy(local_zip, DRIVE_ZIP)
        zip_to_use = local_zip
    else:
        print('Using cached dataset zip from Drive.')
        zip_to_use = DRIVE_ZIP
    os.makedirs(DATA_DIR, exist_ok=True)
    print('Unzipping to local disk...')
    subprocess.run(['unzip', '-q', '-o', zip_to_use, '-d', DATA_DIR], check=True)
print('data ready at', DATA_DIR)
subprocess.run(['ls', DATA_DIR])

In [ ]:
# ── Cell 5: Clone (or sync) the repo; results persist to Drive ──────────────
import subprocess, sys, os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# Persist results to Drive so the run resumes across sessions (skip-guard).
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/thesis_results'

REPO = 'https://github.com/AndreiBrehuescu/action-recognition-thesis.git'
WORK = '/content/thesis'
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, WORK], check=True)
else:
    subprocess.run(['git', '-C', WORK, 'fetch', '--depth', '1', 'origin', 'master'], check=True)
    subprocess.run(['git', '-C', WORK, 'reset', '--hard', 'origin/master'], check=True)
os.chdir(WORK)
subprocess.run(['git', '-C', WORK, 'log', '-1', '--oneline'])
print('repo ready at', WORK, '| results ->', os.environ['RESULTS_DIR'])

In [ ]:
# ── Cell 6: Run the whole pipeline (train -> benchmark -> Pareto plot) ───────
# All settings live in run_all.py; edit there (in the repo), not here.
# On a 24 GB+ GPU (A100/L4) you can raise --batch-size; the code caps to 8 only
# on GPUs under 18 GB. Resumable: finished models are skipped on re-run.
import subprocess, sys
subprocess.run([sys.executable, 'run_all.py',
    '--data-root', '/content/ucf101',
    '--dataset', 'ucf101',
    '--epochs', '10',
    '--batch-size', '8',
], check=True)

In [ ]:
# ── Cell 7: Show the Pareto plot ────────────────────────────────────────────
from IPython.display import Image, display
display(Image('/content/drive/MyDrive/thesis_results/pareto_ucf101.png'))